# Query 9: Transactions

Run `ddl.ipynb` and `dml.ipynb` first. This notebook introduces `BEGIN`, `COMMIT`, and `ROLLBACK` using a temporary copy of one food order.

## Concepts

- A transaction groups changes into one unit of work.
- `COMMIT` keeps the changes; `ROLLBACK` cancels uncommitted changes.
- The example below reads one delivered order from the database into a temporary table, then changes only that copy. The source `orders` table is not modified.

In [ ]:
import sqlite3

connection = sqlite3.connect("file:food-analysis/food_analysis.db?mode=ro", uri=True)
try:
    connection.execute(
        "CREATE TEMP TABLE demo_order AS "
        "SELECT order_id, status FROM orders "
        "WHERE status = 'delivered' ORDER BY order_id LIMIT 1"
    )
    order_id, original_status = connection.execute(
        "SELECT order_id, status FROM demo_order"
    ).fetchone()

    connection.execute("BEGIN")
    connection.execute(
        "UPDATE demo_order SET status = 'pending' WHERE order_id = ?",
        (order_id,),
    )
    connection.commit()
    committed_status = connection.execute(
        "SELECT status FROM demo_order WHERE order_id = ?", (order_id,)
    ).fetchone()[0]

    connection.execute("BEGIN")
    connection.execute(
        "UPDATE demo_order SET status = 'cancelled' WHERE order_id = ?",
        (order_id,),
    )
    connection.rollback()
    rolled_back_status = connection.execute(
        "SELECT status FROM demo_order WHERE order_id = ?", (order_id,)
    ).fetchone()[0]

    print(f"Order copied for practice: {order_id}")
    print(f"Original status: {original_status}")
    print(f"Status after COMMIT: {committed_status}")
    print(f"Status after ROLLBACK: {rolled_back_status}")
finally:
    connection.close()